In [1]:
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
validation_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data

In [ ]:
train_data = train_data.sample(n=10000,random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=700,random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

# Data pre-processing

In [ ]:
def clean_data(text):
  text = re.sub(r"\r\n"," ",text) # lines
  text = re.sub(r"\s+"," ",text) # speac
  text = re.sub(r"<.*?>"," ",text) # html tags
  text = text.strip().lower()

  return text

In [ ]:
train_data['dialogue'] = train_data['dialogue'].fillna('').apply(clean_data)
train_data['summary'] = train_data['summary'].fillna('').apply(clean_data)


validation_data['dialogue'] = validation_data['dialogue'].fillna('').apply(clean_data)
validation_data['summary'] = validation_data['summary'].fillna('').apply(clean_data)

In [ ]:
train_data

# tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
def tokeniz(data):
  inputs = tokenizer(data['dialogue'],padding='max_length',max_length=512, truncation=True)
  target = tokenizer(data['summary'],padding='max_length',max_length=200, truncation=True)

  inputs['labels'] = target['input_ids']
  return inputs

train_dataset = train_data.apply(tokeniz,axis=1).tolist()
validation_dataset = validation_data.apply(tokeniz,axis=1).tolist()

In [ ]:
train_dataset[0]

# Let's build the model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=10,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy ='epoch',
    warmup_steps = 500
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

# Train the model

In [ ]:
trainer.train()

# Save the model

In [ ]:
model.save_pretrained("./save_summary_model")
tokenizer.save_pretrained('./save_summary_model')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./save_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./save_summary_model")

# Testing the model

In [ ]:
def summary(dialogue):
  dialogue=clean_data(dialogue)

  inputs = tokenizer(
       dialogue,
       padding='max_length',
       max_length=512,
       truncation=True,
       return_tensors = "pt"
   )

  targets = model.generate(
       input_ids = inputs["input_ids"],
       attention_mask = inputs['attention_mask'],
       max_length =200,
       num_beams = 5,
       early_stopping=True
   )

  text_summary = tokenizer.decode(targets[0],skip_special_tokens=True)

  return text_summary

In [ ]:
test_dialogue = """Virat Kohli is one of the most well-known and successful cricketers of modern times. He was born on November 5, 1988, in Delhi, India. From a young age, Kohli showed a strong passion for cricket and worked extremely hard to develop his skills. His dedication and determination helped him become one of the leading batsmen in international cricket.

Kohli rose to international fame when he led the Indian Under-19 cricket team to victory in the 2008 Under-19 Cricket World Cup. Soon after, he made his debut for the Indian senior cricket team. He became famous for his aggressive batting style, excellent fitness, consistency, and strong leadership qualities.

As a batsman, Kohli has achieved numerous records across different formats of cricket. He has been particularly successful in One Day International cricket, where he has scored a large number of runs and centuries. His ability to chase difficult targets has earned him a reputation as one of the finest run chasers in cricket.

Kohli also served as captain of the Indian cricket team. Under his leadership, India achieved several important victories, particularly in Test cricket. His emphasis on fitness and discipline also influenced the team's overall approach to professional cricket.

Apart from his cricketing skills, Kohli is known for his competitive personality and commitment to fitness. He has inspired many young athletes to focus on discipline, hard work, and healthy lifestyles.

Virat Kohli has received numerous awards and honors for his contribution to cricket. His journey from a young cricket enthusiast in Delhi to an international cricket superstar is an inspiring example of dedication and perseverance.

In conclusion, Virat Kohli is not only a talented cricketer but also an inspiration to millions of people around the world. His passion, discipline, and determination have made him an important figure in modern cricket."""

In [ ]:
summarizer = summary(test_dialogue)

print(f"Summary : {summarizer}")